# Lab 3 - Trenowanie sieci w Keras: klasyfikacja, regresja i regularyzacja

> **Wersja dla studentów.** Komórki z `# TODO` uzupełnij samodzielnie.

**Kurs:** Uczenie głębokie (Informatyka, II stopień, sem. III, studia niestacjonarne)
**Prowadząca:** dr inż. Natalia Piórkowska
**Środowisko:** Google Colab, GPU T4, TensorFlow / Keras 3

## Cel zajęć

Na poprzednich zajęciach (L1, L2) budowaliśmy sieci "od zera" - liczyliśmy gradienty ręcznie, pisaliśmy własne pętle treningowe. Dziś przechodzimy na poziom, na którym pracuje się na co dzień: **wysokopoziomowe API Keras**. Nauczysz się:

1. budować i trenować sieć MLP do **klasyfikacji wieloklasowej** (Fashion-MNIST) z warstwą softmax i stratą `sparse_categorical_crossentropy`,
2. czytać **krzywe uczenia** i diagnozować niedouczenie / przeuczenie,
3. budować sieć do **regresji** (California Housing), poprawnie standaryzować dane (tylko na zbiorze treningowym!) i porównywać się z **baseline'em**,
4. używać **callbacków**: `EarlyStopping`, `ModelCheckpoint`, `ReduceLROnPlateau`,
5. porównywać **optymalizatory** (SGD, SGD+momentum, RMSprop, Adam) i dobierać **learning rate**,
6. celowo **przeuczyć** sieć, a potem ją "leczyć": Dropout, regularyzacja L2, BatchNormalization,
7. **zapisywać i wczytywać** modele w formacie `.keras`.

## Efekty uczenia (z karty przedmiotu)

- **UGŁ_W01** - zna metody uczenia sieci (funkcje straty, optymalizatory, regularyzacja).
- **UGŁ_W02** - zna zaawansowane narzędzia DL (Keras, callbacki, zapis modeli - element cyklu życia modelu).
- **UGŁ_U01** - formułuje i weryfikuje hipotezy (np. "Dropout zmniejszy różnicę między train a val").
- **UGŁ_U02** - proponuje usprawnienia rozwiązania w oparciu o istniejące biblioteki.

## Powiązanie z wykładem

- **W1** - parametry vs hiperparametry, podział train/val/test, overfitting/underfitting.
- **W2** - warstwa gęsta, aktywacje, softmax, BatchNorm, Dropout.
- **W3** - funkcje straty (MSE, MAE, CCE), optymalizatory (SGD, momentum, RMSProp, Adam), regularyzacja, krzywe uczenia, Keras: `compile` / `fit` / `evaluate`, callbacki. **To główny wykład do tych zajęć.**
- **W4** - metryki klasyfikacji i regresji (zapowiedź - pełna analiza w L5).

## Harmonogram (180 min)

| Czas | Blok |
|---|---|
| 0-10 min | Wstęp, przygotowanie środowiska (sekcja 0) |
| 10-45 min | 1. Klasyfikacja Fashion-MNIST: MLP, softmax, krzywe uczenia (zad. 1.1, 1.2) |
| 45-75 min | 2. Regresja California Housing: standaryzacja, baseline, MSE/MAE (zad. 2.1, 2.2) |
| 75-95 min | 3. Callbacki (zad. 3.1) |
| 95-105 min | **Przerwa (10 min)** |
| 105-130 min | 4. Optymalizatory i learning rate (zad. 4.1, 4.2) |
| 130-160 min | 5. Przeuczenie i regularyzacja: Dropout, L2, BatchNorm (zad. 5.1, 5.2) |
| 160-170 min | 6. Zapis i wczytanie modelu (zad. 6.1) |
| 170-180 min | Pytania kontrolne, podsumowanie, informacje do sprawozdania |

## Wymagania wstępne

- Python, NumPy, matplotlib na poziomie L1-L2.
- Rozumienie: czym jest funkcja straty, gradient, epoka, batch (L1), jak działa MLP i backpropagation (L2).
- Znajomość podziału na zbiór treningowy, walidacyjny i testowy (W1).

**Wersja lokalna (VS Code):** Nie korzystamy z Colaba. Wybierz kernel z jednego wspólnego środowiska, np. `..\Lab02\.venv\Scripts\python.exe`. Do testu ustaw `DL_FAST=1` przed uruchomieniem notebooka. W trybie pełnym (CPU) trening może potrwać długo. Wykresy trafią do `wyniki/wykresy`, pliki modeli do `wyniki/modele`.

## 0. Przygotowanie środowiska

### Włączenie GPU w Colab
W menu wybierz: **Środowisko wykonawcze -> Zmień typ środowiska wykonawczego -> T4 GPU -> Zapisz**. Po zmianie środowisko zostanie zrestartowane, więc zrób to **przed** uruchomieniem pierwszej komórki.

### Tryb szybki
Zmienna `SZYBKI_TRYB` steruje "budżetem obliczeniowym" notebooka. Gdy `SZYBKI_TRYB = True`:
- używamy tylko części danych (np. 2000 pierwszych obrazów),
- trenujemy tylko przez 2-3 epoki.

Wyniki będą wtedy słabsze, ale cały notebook przejdzie w kilka minut nawet na CPU. **Przełącz na `True`, jeśli pracujesz bez GPU** (np. limit GPU w Colab się wyczerpał) albo chcesz najpierw sprawdzić, czy kod działa. Domyślnie wartość jest odczytywana ze zmiennej środowiskowej `DL_FAST` (na zajęciach w Colab będzie to `False`).

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import tensorflow as tf
except ImportError:
    tf = None
import keras
from keras import layers

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, confusion_matrix

# Tryb szybki: True = mniej danych i epok (praca na CPU / szybki test kodu)
SZYBKI_TRYB = os.environ.get("DL_FAST", "0") == "1"
# SZYBKI_TRYB = True   # <- odkomentuj, jeśli pracujesz bez GPU

print("TensorFlow:", tf.__version__ if tf is not None else "brak", "| Keras:", keras.__version__)
print("Backend Keras:", keras.backend.backend())
print("SZYBKI_TRYB =", SZYBKI_TRYB)

In [ ]:
# Kod do wykresow i raportu. Nic nie wrzucamy do folderu zrodlowego.
import re
import sys
from pathlib import Path

WYNIKI = Path("wyniki")
WYKRESY = WYNIKI / "wykresy"
MODELE = WYNIKI / "modele"
WYKRESY.mkdir(parents=True, exist_ok=True)
MODELE.mkdir(parents=True, exist_ok=True)
NR_WYKRESU = 0
RAPORT = {"tryb": "szybki" if SZYBKI_TRYB else "pełny", "seed": 42}
OSTRZEZENIA = []


def zapisz_wykres():
    global NR_WYKRESU
    NR_WYKRESU += 1
    fig = plt.gcf()
    tytul = fig._suptitle.get_text() if fig._suptitle is not None else ""
    if not tytul:
        tytul = next((a.get_title() for a in fig.axes if a.get_title()), "wykres")
    skrot = re.sub(r"[^a-z0-9]+", "_", tytul.lower()).strip("_")[:55]
    sciezka = WYKRESY / f"{NR_WYKRESU:02d}_{skrot or 'wykres'}.png"
    fig.savefig(sciezka, dpi=125, bbox_inches="tight")
    print(f"[WYKRES] {sciezka}")
    plt.show()
    plt.close(fig)


def sprawdz(warunek, opis):
    # Mamy to w logu czarno na bialym; zero udawania, ze przeszlo.
    print(("[OK] " if warunek else "[UWAGA] ") + opis)
    if not warunek:
        OSTRZEZENIA.append(opis)
    return bool(warunek)

In [ ]:
# Sprawdzenie dostępności GPU
gpus = tf.config.list_physical_devices("GPU") if tf is not None else []
if gpus:
    print("Dostępne GPU:", gpus)
else:
    print("Brak GPU - obliczenia na CPU. Rozważ SZYBKI_TRYB = True.")

# Ziarno losowości: ustawia jednocześnie seed dla Pythona, NumPy i TensorFlow.
# Dzięki temu wyniki są (w przybliżeniu) powtarzalne - na GPU drobne różnice i tak mogą wystąpić.
SEED = 42
keras.utils.set_random_seed(SEED)

### Funkcja pomocnicza do rysowania krzywych uczenia

Obiekt `History` zwracany przez `model.fit(...)` ma słownik `history.history`, w którym dla każdej metryki zapisana jest lista wartości z kolejnych epok, np. `loss`, `val_loss`, `accuracy`, `val_accuracy`. Napiszmy funkcję, której będziemy używać przez cały notebook.

In [ ]:
def rysuj_historie(historia, metryka="accuracy", tytul=""):
    # Rysuje stratę i wybraną metrykę dla zbioru treningowego i walidacyjnego
    h = historia.history if hasattr(historia, "history") else historia
    epoki = np.arange(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(epoki, h["loss"], "o-", label="train")
    if "val_loss" in h:
        ax[0].plot(epoki, h["val_loss"], "s-", label="walidacja")
    ax[0].set_xlabel("epoka"); ax[0].set_ylabel("strata"); ax[0].set_title(f"{tytul} - strata")
    ax[0].legend(); ax[0].grid(alpha=0.3)
    if metryka in h:
        ax[1].plot(epoki, h[metryka], "o-", label="train")
        if "val_" + metryka in h:
            ax[1].plot(epoki, h["val_" + metryka], "s-", label="walidacja")
        ax[1].set_xlabel("epoka"); ax[1].set_ylabel(metryka); ax[1].set_title(f"{tytul} - {metryka}")
        ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout(); zapisz_wykres()

## 1. Klasyfikacja wieloklasowa: Fashion-MNIST

### Zbiór danych
Fashion-MNIST to 70 000 obrazów w skali szarości 28x28 przedstawiających 10 kategorii ubrań (60 000 treningowych, 10 000 testowych). Jest trudniejszy niż klasyczny MNIST (cyfry), dlatego lepiej pokazuje różnice między modelami.

### Podział danych - przypomnienie z W1
- **train** - na nim model uczy się parametrów (wag),
- **walidacja (val)** - na nim wybieramy hiperparametry i decydujemy, kiedy przerwać trening,
- **test** - używamy go **raz, na końcu**, do uczciwej oceny. Jeśli patrzymy na test przy dobieraniu hiperparametrów, wynik testowy staje się optymistycznie obciążony.

Keras dostarcza tylko train i test, więc walidację wydzielimy sami z części treningowej.

In [ ]:
(x_tr_full, y_tr_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

if SZYBKI_TRYB:
    # w trybie szybkim bierzemy tylko początkowe próbki
    x_tr_full, y_tr_full = x_tr_full[:2000], y_tr_full[:2000]
    x_test, y_test = x_test[:500], y_test[:500]

# Normalizacja do [0, 1]: sieci uczą się znacznie stabilniej na małych wartościach niż na 0..255
x_tr_full = x_tr_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Wydzielamy walidację: ostatnia 1/6 części treningowej (w pełnym trybie 10 000 obrazów)
n_val = len(x_tr_full) // 6
x_train, y_train = x_tr_full[:-n_val], y_tr_full[:-n_val]
x_val, y_val = x_tr_full[-n_val:], y_tr_full[-n_val:]

NAZWY_KLAS = ["T-shirt/top", "Spodnie", "Sweter", "Sukienka", "Płaszcz",
              "Sandał", "Koszula", "Sneaker", "Torba", "Botek"]

print("train:", x_train.shape, "val:", x_val.shape, "test:", x_test.shape)
print("zakres wartości:", x_train.min(), "-", x_train.max())
print("liczność klas w train:", np.bincount(y_train, minlength=10))

In [ ]:
# Podgląd kilku obrazów z etykietami
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(NAZWY_KLAS[y_train[i]], fontsize=9)
    ax.axis("off")
plt.suptitle("Przykładowe obrazy Fashion-MNIST (zbiór treningowy)")
plt.tight_layout(); zapisz_wykres()

### Softmax i entropia krzyżowa

Ostatnia warstwa klasyfikatora na $K$ klas ma $K$ neuronów zwracających liczby rzeczywiste $z_1, \dots, z_K$ (tzw. **logity**). Funkcja **softmax** zamienia je na rozkład prawdopodobieństwa:

$$p_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}, \qquad \sum_k p_k = 1.$$

Funkcja straty **kategorialnej entropii krzyżowej** dla jednej próbki z prawdziwą klasą $y$ to

$$L = -\log p_y.$$

Intuicja: jeśli model daje prawdziwej klasie prawdopodobieństwo bliskie 1, strata jest bliska 0; jeśli bliskie 0, strata rośnie do nieskończoności - model jest mocno karany za pewne, ale błędne odpowiedzi.

### `sparse_categorical_crossentropy` vs `categorical_crossentropy`
- `categorical_crossentropy` - etykiety w postaci **one-hot**, np. klasa 3 -> `[0,0,0,1,0,0,0,0,0,0]`,
- `sparse_categorical_crossentropy` - etykiety jako **liczby całkowite** (0..9). Matematycznie to ta sama strata, ale nie trzeba tworzyć macierzy one-hot.

Nasze `y_train` to liczby całkowite, więc użyjemy wersji `sparse_`. **Typowy błąd:** użycie `categorical_crossentropy` z etykietami całkowitymi - Keras zgłosi błąd niezgodności kształtów.

### Ile parametrów ma warstwa gęsta? (W2)
Warstwa `Dense(n_out)` przy wejściu o wymiarze $n_{in}$ ma $n_{in} \cdot n_{out} + n_{out}$ parametrów (wagi + biasy). Dla `Flatten` 28x28 = 784 wejść i 128 neuronów: $784 \cdot 128 + 128 = 100\,480$.

In [ ]:
# Model demonstracyjny: najprostszy MLP z jedną warstwą ukrytą
model_demo = keras.Sequential([
    keras.Input(shape=(28, 28)),            # deklaracja kształtu wejścia (bez wymiaru batcha)
    layers.Flatten(),                       # 28x28 -> wektor 784
    layers.Dense(128, activation="relu"),   # warstwa ukryta
    layers.Dense(10, activation="softmax"), # 10 klas -> rozkład prawdopodobieństwa
], name="mlp_demo")

model_demo.summary()   # sprawdź: 100 480 + 1 290 = 101 770 parametrów

### `compile` i `fit`

- `compile(optimizer, loss, metrics)` - konfiguruje trening: czym optymalizujemy, co minimalizujemy i co dodatkowo raportujemy.
- `fit(x, y, epochs, batch_size, validation_data)` - właściwy trening. W każdej epoce model przechodzi raz przez cały zbiór treningowy w porcjach (mini-batchach) po `batch_size` próbek. Po każdej epoce liczone są metryki na zbiorze walidacyjnym.
- `evaluate(x, y)` - liczy stratę i metryki na podanym zbiorze, bez uczenia.

In [ ]:
EPOKI_DEMO = 2 if SZYBKI_TRYB else 10

model_demo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

t0 = time.time()
hist_demo = model_demo.fit(
    x_train, y_train,
    epochs=EPOKI_DEMO,
    batch_size=128,
    validation_data=(x_val, y_val),   # walidacja liczona po każdej epoce
    verbose=2,                        # jedna linia na epokę
)
print(f"czas treningu: {time.time() - t0:.1f} s")
rysuj_historie(hist_demo, tytul="MLP demo")

### Jak czytać krzywe uczenia (W3)?

| Obserwacja | Diagnoza | Co zrobić |
|---|---|---|
| strata train i val wysokie, zbliżone | **niedouczenie** (underfitting) | większy model, dłuższy trening, większy lr |
| strata train maleje, val maleje i stabilizuje się | dobre dopasowanie | ewentualnie zatrzymać w minimum val |
| strata train maleje, val zaczyna **rosnąć** | **przeuczenie** (overfitting) | regularyzacja, więcej danych, early stopping |
| strata skacze / rośnie od początku | za duży learning rate, błąd w danych | zmniejszyć lr, sprawdzić dane |

Uwaga: w pierwszych epokach dokładność walidacyjna bywa **wyższa** niż treningowa. To nie błąd - metryki treningowe są uśredniane w trakcie epoki (model na początku epoki był gorszy), a walidacyjne liczone na końcu epoki.

### Zadanie 1.1 - Głębszy MLP

Zbuduj model `model_mlp` (Sequential) o architekturze:
`Input(28,28) -> Flatten -> Dense(256, relu) -> Dense(128, relu) -> Dense(10, softmax)`.

1. Skompiluj go z optymalizatorem Adam (lr = 1e-3), stratą `sparse_categorical_crossentropy` i metryką `accuracy`.
2. Wytrenuj przez `EPOKI_11` epok z `batch_size=128`, z walidacją na `(x_val, y_val)`. Historię zapisz w `hist_mlp`.
3. Narysuj krzywe uczenia funkcją `rysuj_historie`.
4. Oceń model na zbiorze **testowym** (`evaluate`) i wypisz dokładność.
5. Ręcznie policz liczbę parametrów i porównaj z `model_mlp.count_params()`.

**Kryteria sukcesu (pełny tryb, GPU):** dokładność walidacyjna >= 0.87, testowa >= 0.86; liczba parametrów zgadza się z Twoim rachunkiem.

**Wskazówki:** wzorem jest `model_demo` powyżej. Parametry: $784\cdot256+256$, $256\cdot128+128$, $128\cdot10+10$.

In [ ]:
EPOKI_11 = 2 if SZYBKI_TRYB else 15
# 1.1. Dokladnie architektura z polecenia. Na wyjsciu softmax, bez kombinacji.
keras.utils.set_random_seed(SEED)
model_mlp = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
], name="mlp_fashion_256_128")
model_mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
hist_mlp = model_mlp.fit(
    x_train, y_train, epochs=EPOKI_11, batch_size=128,
    validation_data=(x_val, y_val), verbose=2,
)
rysuj_historie(hist_mlp, tytul="1.1 MLP 256-128")
loss_test_mlp, acc_test_mlp = model_mlp.evaluate(x_test, y_test, verbose=0)
liczba_param_recznie = (784*256+256) + (256*128+128) + (128*10+10)
print(f"MLP - val acc={hist_mlp.history['val_accuracy'][-1]:.3f}, test acc={acc_test_mlp:.3f}")
print("parametry ręcznie / Keras:", liczba_param_recznie, model_mlp.count_params())
sprawdz(liczba_param_recznie == model_mlp.count_params(), "1.1: liczba parametrow zgodna")
RAPORT["zad_1_1"] = {
    "val_accuracy_last": float(hist_mlp.history["val_accuracy"][-1]),
    "test_accuracy": float(acc_test_mlp), "test_loss": float(loss_test_mlp),
    "parametry": int(model_mlp.count_params()),
}


### Gdzie model się myli? Macierz pomyłek

Sama dokładność to za mało - chcemy wiedzieć, **które klasy** są mylone. Macierz pomyłek $C$ ma w komórce $C_{ij}$ liczbę próbek o prawdziwej klasie $i$, które model przypisał do klasy $j$. Na przekątnej są poprawne odpowiedzi. (Pełna analiza metryk klasyfikacji: W4 i L5.)

`model.predict(x)` zwraca macierz prawdopodobieństw o kształcie `(n, 10)`; klasę przewidywaną wybieramy przez `argmax` po osi 1.

In [ ]:
# Demonstracja na model_demo: predykcja i macierz pomyłek
proby = model_demo.predict(x_test, verbose=0)     # (n, 10) prawdopodobieństwa
y_pred_demo = proby.argmax(axis=1)                # klasa o największym prawdopodobieństwie
print("suma prawdopodobieństw w 1. wierszu:", proby[0].sum().round(4))  # softmax -> 1.0

cm = confusion_matrix(y_test, y_pred_demo, labels=range(10))
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(NAZWY_KLAS, rotation=60, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(NAZWY_KLAS)
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("klasa przewidziana"); ax.set_ylabel("klasa prawdziwa")
ax.set_title("Macierz pomyłek - MLP demo (test)")
plt.colorbar(im, ax=ax, label="liczba próbek"); plt.tight_layout(); zapisz_wykres()

### Zadanie 1.2 - Analiza błędów

Dla modelu `model_mlp` z zadania 1.1:

1. Wyznacz predykcje na zbiorze testowym (`y_pred_mlp`).
2. Policz **dokładność dla każdej klasy osobno** (odsetek poprawnych odpowiedzi wśród próbek danej klasy) i przedstaw ją na wykresie słupkowym z nazwami klas.
3. Znajdź **parę klas najczęściej mylonych** (największy element macierzy pomyłek poza przekątną) i wypisz ją.
4. Pokaż 10 błędnie sklasyfikowanych obrazów z podpisem "prawda / predykcja (prawdopodobieństwo)".

**Kryteria sukcesu:** wykres ma tytuł i opisane osie; potrafisz wyjaśnić, dlaczego właśnie te klasy są mylone.

**Wskazówki:** dokładność klasy $k$ to `cm[k, k] / cm[k].sum()`. Żeby znaleźć maksimum poza przekątną, skopiuj macierz i wyzeruj przekątną (`np.fill_diagonal`), potem `np.unravel_index(m.argmax(), m.shape)`.

In [ ]:
# 1.2. Prawdopodobienstwa -> indeks klasy. Bez sumowania po wierszach.
p_mlp = model_mlp.predict(x_test, verbose=0)
y_pred_mlp = np.argmax(p_mlp, axis=1)
cm_mlp = confusion_matrix(y_test, y_pred_mlp, labels=range(10))
acc_klas = np.diag(cm_mlp) / np.maximum(cm_mlp.sum(axis=1), 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(NAZWY_KLAS, acc_klas)
ax.set_ylim(0, 1.0)
ax.tick_params(axis="x", labelrotation=45)
ax.set_ylabel("dokładność klasy")
ax.set_xlabel("klasa")
ax.set_title("1.2 Dokładność na klasę - MLP")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout(); zapisz_wykres()

bez_przekatnej = cm_mlp.copy()
np.fill_diagonal(bez_przekatnej, 0)
klasa_prawda, klasa_mylona = np.unravel_index(np.argmax(bez_przekatnej), bez_przekatnej.shape)
print("Najczesciej mylona para:", NAZWY_KLAS[klasa_prawda], "->", NAZWY_KLAS[klasa_mylona],
      "liczba:", bez_przekatnej[klasa_prawda, klasa_mylona])

# Tu celowo wybieram bledy. Nie tylko ladne przyklady.
bledne_indeksy = np.flatnonzero(y_pred_mlp != y_test)
fig, axes = plt.subplots(2, 5, figsize=(13, 6))
for ax, idx in zip(axes.flat, bledne_indeksy[:10]):
    ax.imshow(x_test[idx], cmap="gray")
    conf = p_mlp[idx, y_pred_mlp[idx]]
    ax.set_title(f"{NAZWY_KLAS[y_test[idx]]} / {NAZWY_KLAS[y_pred_mlp[idx]]}\n({conf:.0%})", fontsize=8)
    ax.axis("off")
for ax in list(axes.flat)[min(10, len(bledne_indeksy)):]:
    ax.axis("off")
fig.suptitle("1.2 Dziesięć błędnych predykcji: prawda / model")
plt.tight_layout(); zapisz_wykres()
RAPORT["zad_1_2"] = {
    "accuracies_per_class": {NAZWY_KLAS[i]: round(float(a), 4) for i, a in enumerate(acc_klas)},
    "most_confused_pair": [NAZWY_KLAS[klasa_prawda], NAZWY_KLAS[klasa_mylona]],
    "most_confused_count": int(bez_przekatnej[klasa_prawda, klasa_mylona]),
}


## 2. Regresja: California Housing

### Zbiór danych
Dane ze spisu powszechnego w Kalifornii (1990): 20 640 dzielnic, 8 cech numerycznych (mediana dochodu, wiek budynków, średnia liczba pokoi, populacja, położenie geograficzne...). Cel: **mediana wartości domu** w dzielnicy, w setkach tysięcy dolarów.

### Co zmienia się względem klasyfikacji?
- warstwa wyjściowa: **1 neuron bez aktywacji** (aktywacja liniowa) - wyjście może być dowolną liczbą rzeczywistą,
- funkcja straty: **MSE** $= \frac{1}{n}\sum_i (y_i - \hat y_i)^2$ lub **MAE** $= \frac{1}{n}\sum_i |y_i - \hat y_i|$ (W3),
- metryki: MAE (w jednostkach celu, łatwa interpretacja), RMSE $= \sqrt{\text{MSE}}$, ewentualnie $R^2$.

MSE silniej karze duże błędy (kwadrat), MAE jest bardziej odporne na wartości odstające. Często trenujemy na MSE (gładki gradient), a raportujemy MAE.

In [ ]:
dane = fetch_california_housing(as_frame=True)
df = dane.frame
if SZYBKI_TRYB:
    df = df.iloc[:2000]
print(df.shape)
df.describe().T.round(2)

### Standaryzacja - tylko na zbiorze treningowym!

Cechy mają bardzo różne skale (populacja - tysiące, mediana dochodu - kilka jednostek). Gradient względem wag przy cechach o dużej skali byłby ogromny, a przy małej - znikomy; trening byłby niestabilny. Dlatego standaryzujemy:

$$x' = \frac{x - \mu_{train}}{\sigma_{train}}.$$

**Kluczowe:** średnią $\mu$ i odchylenie $\sigma$ liczymy **wyłącznie na zbiorze treningowym**, a potem tymi samymi wartościami przekształcamy walidację i test. Gdybyśmy dopasowali `StandardScaler` do całych danych, informacja ze zbioru testowego "przeciekłaby" do treningu (**data leakage**, W4) - wynik testowy byłby zbyt optymistyczny. W praktyce produkcyjnej nowe dane też przekształcamy parametrami z treningu, bo przyszłości nie znamy.

Schemat: `scaler.fit(X_train)` -> `scaler.transform(X_train / X_val / X_test)`.

In [ ]:
X = df[dane.feature_names].values.astype("float32")
y = df["MedHouseVal"].values.astype("float32")

# Podział 60 / 20 / 20: najpierw odcinamy test, potem z reszty walidację
X_tmp, X_te_r, y_tmp, y_te_r = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_tr_r, X_va_r, y_tr_r, y_va_r = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=SEED)

scaler = StandardScaler().fit(X_tr_r)       # fit TYLKO na train
X_tr_s = scaler.transform(X_tr_r)           # transform na wszystkich podzbiorach
X_va_s = scaler.transform(X_va_r)
X_te_s = scaler.transform(X_te_r)

print("train/val/test:", X_tr_s.shape, X_va_s.shape, X_te_s.shape)
print("średnie cech po standaryzacji (train):", X_tr_s.mean(axis=0).round(3))
print("średnie cech po standaryzacji (test): ", X_te_s.mean(axis=0).round(3), "<- nie muszą być 0")

### Baseline - zawsze zaczynaj od punktu odniesienia

Zanim powiesz, że sieć "działa dobrze", porównaj ją z czymś prostym. Najprostszy baseline w regresji: **zawsze przewiduj średnią z treningu**. Jeśli sieć nie jest wyraźnie lepsza, coś jest nie tak (dane, skalowanie, architektura).

In [ ]:
# Baseline 0: stała predykcja = średnia celu na zbiorze treningowym
pred_sr = np.full_like(y_te_r, y_tr_r.mean())
mae_sr = mean_absolute_error(y_te_r, pred_sr)
rmse_sr = np.sqrt(mean_squared_error(y_te_r, pred_sr))
print(f"Baseline (średnia): MAE = {mae_sr:.3f}, RMSE = {rmse_sr:.3f}  (jednostka: 100 tys. $)")

### Zadanie 2.1 - Baseline liniowy

Mocniejszym punktem odniesienia jest **regresja liniowa** (klasyczny model ML, W1).

1. Dopasuj `LinearRegression` ze `sklearn` do `(X_tr_s, y_tr_r)`.
2. Policz MAE i RMSE na zbiorze **testowym**; zapisz w zmiennych `mae_lin`, `rmse_lin`.
3. Wypisz porównanie z baseline'em średniej.

**Kryteria sukcesu (pełny tryb):** MAE regresji liniowej ok. 0.53, wyraźnie mniej niż baseline średniej (ok. 0.9).

In [ ]:
# 2.1. Porownanie na tej samej probie testowej - inaczej metryki nic nie mowia.
model_liniowy = LinearRegression()
model_liniowy.fit(X_tr_s, y_tr_r)
pred_lin = model_liniowy.predict(X_te_s)
mae_lin = mean_absolute_error(y_te_r, pred_lin)
rmse_lin = np.sqrt(mean_squared_error(y_te_r, pred_lin))
print(f"Średnia: MAE={mae_sr:.3f}, RMSE={rmse_sr:.3f}")
print(f"Linear:  MAE={mae_lin:.3f}, RMSE={rmse_lin:.3f}")
sprawdz(mae_lin < mae_sr, "2.1: regresja liniowa bije baseline ze sredniej")
RAPORT["zad_2_1"] = {"mean_mae": float(mae_sr), "mean_rmse": float(rmse_sr),
                      "linear_mae": float(mae_lin), "linear_rmse": float(rmse_lin)}


### Zadanie 2.2 - Sieć do regresji

1. Zbuduj `model_reg`: `Input(8) -> Dense(64, relu) -> Dense(64, relu) -> Dense(1)` (bez aktywacji na wyjściu!).
2. Skompiluj z Adam (lr = 1e-3), stratą `"mse"` i metryką `"mae"`.
3. Wytrenuj `EPOKI_22` epok, `batch_size=64`, z walidacją `(X_va_s, y_va_r)`. Narysuj krzywe (`metryka="mae"`).
4. Policz MAE i RMSE na teście (`mae_nn`, `rmse_nn`) i zestaw w tabeli (pandas DataFrame) z baseline'ami.
5. Narysuj wykres rozrzutu: wartości prawdziwe vs przewidziane na teście, z linią $y = x$.

**Kryteria sukcesu (pełny tryb):** MAE sieci < 0.42 (wyraźnie lepiej niż liniowa).

**Wskazówki:** `model_reg.predict(X_te_s).ravel()` - wyjście ma kształt `(n, 1)`, spłaszczamy je do `(n,)`. Zapomnienie o `ravel()` przy liczeniu `y - y_pred` daje broadcasting do macierzy `(n, n)` - klasyczny, cichy błąd!

In [ ]:
EPOKI_22 = 3 if SZYBKI_TRYB else 60
# 2.2. Regresja: ostatnia warstwa MUSI byc liniowa, nie softmax.
keras.utils.set_random_seed(SEED)
model_reg = keras.Sequential([
    keras.Input(shape=(8,)), layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"), layers.Dense(1),
], name="regresja_kalifornia")
model_reg.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
hist_reg = model_reg.fit(X_tr_s, y_tr_r, epochs=EPOKI_22, batch_size=64,
                         validation_data=(X_va_s, y_va_r), verbose=2)
rysuj_historie(hist_reg, metryka="mae", tytul="2.2 Regresja MLP")
pred_nn = model_reg.predict(X_te_s, verbose=0).ravel()
# ravel jest tu wazny. Bez niego broadcasting zrobi macierz N x N.
mae_nn = mean_absolute_error(y_te_r, pred_nn)
rmse_nn = np.sqrt(mean_squared_error(y_te_r, pred_nn))
tabela_reg = pd.DataFrame([
    {"model": "średnia", "MAE": mae_sr, "RMSE": rmse_sr},
    {"model": "liniowy", "MAE": mae_lin, "RMSE": rmse_lin},
    {"model": "MLP 64-64", "MAE": mae_nn, "RMSE": rmse_nn},
])
print(tabela_reg.round(4).to_string(index=False))
tabela_reg.to_csv(WYNIKI / "porownanie_regresji.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_te_r, pred_nn, s=9, alpha=0.28)
zakres = [min(float(np.min(y_te_r)), float(np.min(pred_nn))),
          max(float(np.max(y_te_r)), float(np.max(pred_nn)))]
ax.plot(zakres, zakres, "k--", label="idealnie: y = x")
ax.set(xlabel="wartość rzeczywista (100 tys. USD)",
       ylabel="predykcja (100 tys. USD)", title="2.2 California Housing: y vs pred")
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); zapisz_wykres()
RAPORT["zad_2_2"] = {"mae_nn": float(mae_nn), "rmse_nn": float(rmse_nn),
                       "lepszy_od_liniowego": bool(mae_nn < mae_lin)}


## 3. Callbacki - automatyzacja treningu

**Callback** to obiekt, którego metody Keras wywołuje w określonych momentach treningu (początek/koniec epoki, batcha, treningu). Najważniejsze wbudowane:

| Callback | Co robi | Kluczowe argumenty |
|---|---|---|
| `EarlyStopping` | przerywa trening, gdy monitorowana metryka przestaje się poprawiać | `monitor="val_loss"`, `patience`, `restore_best_weights=True` |
| `ModelCheckpoint` | zapisuje model (lub wagi) na dysk, np. tylko gdy jest najlepszy | `filepath` (`.keras` lub `.weights.h5`), `save_best_only=True` |
| `ReduceLROnPlateau` | zmniejsza learning rate, gdy metryka stoi w miejscu | `factor` (np. 0.5), `patience`, `min_lr` |

**Early stopping jako regularyzacja (W3):** zatrzymując trening w minimum `val_loss`, nie pozwalamy sieci "zapamiętać" szumu ze zbioru treningowego. `patience` mówi, ile epok bez poprawy tolerujemy (krzywa walidacyjna jest zaszumiona, więc 1 epoka to za mało). `restore_best_weights=True` przywraca wagi z najlepszej epoki - bez tego zostałyby wagi z ostatniej (gorszej) epoki.

**ReduceLROnPlateau - intuicja:** na początku duży krok pozwala szybko zejść w dół funkcji straty; blisko minimum duży krok "przeskakuje" dolinę, więc krok zmniejszamy.

In [ ]:
# Demonstracja: EarlyStopping na klasyfikacji Fashion-MNIST z dużą maksymalną liczbą epok
EPOKI_MAX_DEMO = 3 if SZYBKI_TRYB else 50

wczesne_zatrzymanie = keras.callbacks.EarlyStopping(
    monitor="val_loss",          # obserwujemy stratę walidacyjną
    patience=3,                  # 3 epoki bez poprawy -> stop
    restore_best_weights=True,   # przywróć wagi z najlepszej epoki
    verbose=1,
)

model_es = keras.Sequential([keras.Input(shape=(28, 28)), layers.Flatten(),
                             layers.Dense(256, activation="relu"),
                             layers.Dense(10, activation="softmax")])
model_es.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_es = model_es.fit(x_train, y_train, epochs=EPOKI_MAX_DEMO, batch_size=128,
                       validation_data=(x_val, y_val), callbacks=[wczesne_zatrzymanie], verbose=0)

n_epok = len(hist_es.history["loss"])
najlepsza = int(np.argmin(hist_es.history["val_loss"])) + 1
print(f"Wykonano {n_epok} z {EPOKI_MAX_DEMO} epok; najlepsza epoka (min val_loss): {najlepsza}")
rysuj_historie(hist_es, tytul="EarlyStopping")

### Zadanie 3.1 - Trzy callbacki naraz (regresja)

Wytrenuj **nowy** model o architekturze jak w zad. 2.2 (`model_cb`) na danych California Housing, z maksymalną liczbą epok `EPOKI_MAX_31` i trzema callbackami:

- `EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)`,
- `ModelCheckpoint("najlepszy_reg.keras", monitor="val_loss", save_best_only=True)`,
- `ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5)`.

Następnie:
1. wypisz, po ilu epokach trening się zatrzymał i która epoka była najlepsza,
2. narysuj, jak zmieniał się learning rate w kolejnych epokach (Keras zapisuje go w `historia.history["learning_rate"]`),
3. wczytaj model z pliku `najlepszy_reg.keras` i policz jego MAE na teście.

**Kryteria sukcesu:** plik `najlepszy_reg.keras` istnieje; w pełnym trybie trening kończy się przed limitem epok, a lr zmalał co najmniej raz; MAE testowe wczytanego modelu <= MAE z zadania 2.2 (lub bardzo zbliżone).

**Wskazówka:** callbacki przekazujemy jako listę: `fit(..., callbacks=[cb1, cb2, cb3])`.

In [ ]:
EPOKI_MAX_31 = 3 if SZYBKI_TRYB else 200
# 3.1. Trzy callbacki i nowy model. Checkpoint do wynikow, nie do repo.
keras.utils.set_random_seed(SEED)
model_cb = keras.Sequential([
    keras.Input(shape=(8,)), layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"), layers.Dense(1),
], name="regresja_callbacki")
model_cb.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
sciezka_best = MODELE / "najlepszy_reg.keras"
cb_es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
cb_checkpoint = keras.callbacks.ModelCheckpoint(
    str(sciezka_best), monitor="val_loss", save_best_only=True,
)
cb_reduce = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5,
)
hist_cb = model_cb.fit(X_tr_s, y_tr_r, epochs=EPOKI_MAX_31, batch_size=64,
                       validation_data=(X_va_s, y_va_r),
                       callbacks=[cb_es, cb_checkpoint, cb_reduce], verbose=2)
liczba_epok_cb = len(hist_cb.history["loss"])
najlepsza_epoka_cb = int(np.argmin(hist_cb.history["val_loss"])) + 1
print("Callbacki: wykonano", liczba_epok_cb, "epok; najlepsza:", najlepsza_epoka_cb)
lr_hist = hist_cb.history.get("learning_rate", [])
if len(lr_hist) == 0:
    # Starszy Keras czasem nie dodaje tego klucza; uczciwie zostawiam info.
    OSTRZEZENIA.append("Keras nie zapisal learning_rate w historii; brak krzywej lr")
else:
    lr_hist = np.asarray(lr_hist, dtype=float).ravel()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(range(1, len(lr_hist) + 1), lr_hist, "o-")
    ax.set(xlabel="epoka", ylabel="learning rate", title="3.1 ReduceLROnPlateau: przebieg lr")
    ax.grid(alpha=0.25)
    plt.tight_layout(); zapisz_wykres()
model_najlepszy = keras.models.load_model(sciezka_best)
pred_best = model_najlepszy.predict(X_te_s, verbose=0).ravel()
mae_cb = mean_absolute_error(y_te_r, pred_best)
print(f"Checkpoint: test MAE={mae_cb:.4f} (zwykly MLP={mae_nn:.4f})")
sprawdz(sciezka_best.is_file(), "3.1: checkpoint zapisany")
RAPORT["zad_3_1"] = {"epochs": liczba_epok_cb, "best_epoch": najlepsza_epoka_cb,
                       "early_stopped": bool(liczba_epok_cb < EPOKI_MAX_31),
                       "lr_min": float(np.min(lr_hist)) if len(lr_hist) else None,
                       "lr_was_reduced": bool(np.min(lr_hist) < np.max(lr_hist)) if len(lr_hist) else False,
                       "loaded_model_test_mae": float(mae_cb),
                       "checkpoint_exists": bool(sciezka_best.is_file())}


---
## Przerwa (10 min)
---

## 4. Optymalizatory i learning rate

Przypomnienie z W3 (wzory dla jednego parametru $w$, gradient $g_t = \nabla_w L$, learning rate $\eta$):

- **SGD:** $w \leftarrow w - \eta\, g_t$. Prosty, ale wolny w "wąskich dolinach" funkcji straty.
- **SGD z momentum:** $v \leftarrow \beta v - \eta g_t,\; w \leftarrow w + v$ (typowo $\beta = 0.9$). Jak kulka toczona po zboczu - "rozpęd" wygładza oscylacje i przyspiesza ruch w spójnym kierunku.
- **RMSprop:** $s \leftarrow \rho s + (1-\rho) g_t^2,\; w \leftarrow w - \frac{\eta}{\sqrt{s}+\epsilon} g_t$. Każdy parametr ma własny, adaptacyjny krok - duże gradienty są tłumione.
- **Adam:** łączy momentum (średnia gradientów $m$) i RMSprop (średnia kwadratów $v$) z korekcją obciążenia: $w \leftarrow w - \eta\,\frac{\hat m}{\sqrt{\hat v}+\epsilon}$. Bardzo dobry wybór domyślny (lr = 1e-3).

**Learning rate** to zwykle najważniejszy hiperparametr. Za mały - trening bardzo wolny; za duży - strata oscyluje lub rośnie (rozbieżność, jak w L1).

Do porównań użyjemy funkcji, która za każdym razem buduje **identyczny** model od nowa (te same seedy), zmieniając tylko optymalizator - wtedy porównanie jest uczciwe.

In [ ]:
EPOKI_OPT = 2 if SZYBKI_TRYB else 10

def zbuduj_mlp_fashion():
    # Ten sam MLP za każdym razem; seed gwarantuje identyczną inicjalizację wag
    keras.utils.set_random_seed(SEED)
    return keras.Sequential([keras.Input(shape=(28, 28)), layers.Flatten(),
                             layers.Dense(128, activation="relu"),
                             layers.Dense(10, activation="softmax")])

def trenuj_z_optymalizatorem(optymalizator, epoki=EPOKI_OPT):
    # Trenuje świeży model z podanym optymalizatorem i zwraca słownik historii
    model = zbuduj_mlp_fashion()
    model.compile(optimizer=optymalizator, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h = model.fit(x_train, y_train, epochs=epoki, batch_size=128,
                  validation_data=(x_val, y_val), verbose=0)
    return h.history

# Demonstracja: zwykły SGD vs Adam
wyniki_demo = {
    "SGD lr=0.01": trenuj_z_optymalizatorem(keras.optimizers.SGD(learning_rate=0.01)),
    "Adam lr=0.001": trenuj_z_optymalizatorem(keras.optimizers.Adam(learning_rate=0.001)),
}
plt.figure(figsize=(7, 4))
for nazwa, h in wyniki_demo.items():
    plt.plot(np.arange(1, len(h["val_loss"]) + 1), h["val_loss"], "o-", label=nazwa)
plt.xlabel("epoka"); plt.ylabel("val_loss"); plt.title("SGD vs Adam - strata walidacyjna")
plt.legend(); plt.grid(alpha=0.3); zapisz_wykres()

### Zadanie 4.1 - Porównanie czterech optymalizatorów

Używając `trenuj_z_optymalizatorem`, porównaj:
- `SGD(learning_rate=0.01)`,
- `SGD(learning_rate=0.01, momentum=0.9)`,
- `RMSprop(learning_rate=0.001)`,
- `Adam(learning_rate=0.001)`.

1. Zapisz historie w słowniku `wyniki_opt` (klucz = nazwa).
2. Narysuj na **jednym** wykresie `val_loss` wszystkich wariantów, a na drugim `val_accuracy`.
3. Utwórz tabelę (DataFrame) z najlepszą `val_accuracy` i epoką, w której ją osiągnięto.

**Kryteria sukcesu:** SGD bez momentum powinien być wyraźnie najwolniejszy; potrafisz to wyjaśnić na podstawie wzorów powyżej.

**Uwaga:** optymalizator to obiekt ze stanem (np. bufor momentum) - dla każdego modelu twórz **nową** instancję, nie używaj tej samej dwa razy.

In [ ]:
# 4.1. Osobny optymalizator za kazdym razem; Adam ma swoj stan i tyle.
konfiguracje = {
    "SGD": lambda: keras.optimizers.SGD(learning_rate=0.01),
    "SGD+momentum": lambda: keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSprop": lambda: keras.optimizers.RMSprop(learning_rate=0.001),
    "Adam": lambda: keras.optimizers.Adam(learning_rate=0.001),
}
wyniki_opt = {nazwa: trenuj_z_optymalizatorem(f()) for nazwa, f in konfiguracje.items()}
for pole, tytul in [("val_loss", "4.1 Optymalizatory: val_loss"),
                   ("val_accuracy", "4.1 Optymalizatory: val_accuracy")]:
    fig, ax = plt.subplots(figsize=(8, 5))
    for nazwa, h in wyniki_opt.items():
        ax.plot(np.arange(1, len(h[pole])+1), h[pole], "o-", label=nazwa)
    ax.set(xlabel="epoka", ylabel=pole, title=tytul)
    ax.legend(); ax.grid(alpha=0.2)
    plt.tight_layout(); zapisz_wykres()

tabela_opt = pd.DataFrame([
    {"optymalizator": nazwa,
     "best_val_accuracy": float(max(h["val_accuracy"])),
     "best_epoch": int(np.argmax(h["val_accuracy"])) + 1}
    for nazwa, h in wyniki_opt.items()
]).sort_values("best_val_accuracy", ascending=False)
print(tabela_opt.to_string(index=False))
tabela_opt.to_csv(WYNIKI / "optymalizatory.csv", index=False)
RAPORT["zad_4_1"] = tabela_opt.to_dict(orient="records")


### Zadanie 4.2 - Wpływ learning rate

Dla optymalizatora Adam sprawdź learning rate z listy `[1e-4, 1e-3, 1e-2, 1e-1]`.

1. Narysuj krzywe `loss` (treningowej!) dla wszystkich wartości na jednym wykresie.
2. Opisz w komórce markdown (2-3 zdania): który lr jest za mały, który za duży i po czym to widać.

**Kryteria sukcesu:** dla lr = 0.1 widać wyraźnie gorszą (wyższą, niestabilną) stratę, dla 1e-4 - powolny spadek.

**Wskazówka:** jeśli strata dla dużego lr robi się `nan`, to też jest wynik - zapisz to w wnioskach (rozbieżność).

In [ ]:
# 4.2. Na LR 0.1 wynik moze byc kiepski - to czesc eksperymentu.
wartosci_lr = [1e-4, 1e-3, 1e-2, 1e-1]
wyniki_lr = {}
for lr in wartosci_lr:
    print("Adam LR:", lr)
    wyniki_lr[lr] = trenuj_z_optymalizatorem(
        keras.optimizers.Adam(learning_rate=lr))
fig, ax = plt.subplots(figsize=(9, 5))
for lr, h in wyniki_lr.items():
    ax.plot(np.arange(1, len(h["loss"])+1), h["loss"], "o-", label=f"lr={lr:g}")
ax.set(xlabel="epoka", ylabel="loss (trening)", title="4.2 Adam - wpływ learning rate")
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); zapisz_wykres()
RAPORT["zad_4_2"] = {
    f"{lr:g}": {
        "loss_last": float(h["loss"][-1]) if np.isfinite(h["loss"][-1]) else None,
        "loss_min": float(np.nanmin(h["loss"])) if np.isfinite(h["loss"]).any() else None,
    } for lr, h in wyniki_lr.items()
}
if RAPORT["zad_4_2"]["0.1"]["loss_last"] is None:
    print("LR=0.1 -> NaN/inf: to tez wynik. Spisuje zamiast ukrywac.")
print("LR = 1e-4: typowo wolne schodzenie z loss. LR = 0.1: sprawdzac skoki/straty,")
print("nie zakladam z gory rozbieznosci - patrz konkretne krzywe.")


*Wnioski (4.2):* Mały LR (1e-4) zwykle oznacza wolną zbieżność. LR=1e-1 może podbić stratę i powodować oscylacje, jednak ostateczną diagnozę opieram na wygenerowanym wykresie i wartościach `wyniki/podsumowanie.json`.


## 5. Celowe przeuczenie i jego "leczenie"

### Przeuczenie (overfitting)
Model o dużej pojemności (dużo parametrów) trenowany na małym zbiorze potrafi "zapamiętać" dane treningowe łącznie z szumem. Objaw: strata treningowa spada niemal do zera, a walidacyjna po pewnym czasie **rośnie**. Żeby wyraźnie to zobaczyć, celowo:
- zmniejszymy zbiór treningowy (do 1000 obrazów),
- użyjemy dużej sieci (3 x 512 neuronów, ok. 930 tys. parametrów),
- będziemy trenować długo.

### Metody regularyzacji (W2, W3)
- **Dropout($p$):** w trakcie treningu każdy neuron jest losowo "wyłączany" z prawdopodobieństwem $p$ (pozostałe są skalowane przez $1/(1-p)$). Sieć nie może polegać na pojedynczych neuronach - uczy się reprezentacji redundantnych. Działa jak uśrednianie wielu "podsieci". W predykcji Dropout jest wyłączony.
- **L2 (weight decay):** do straty dodajemy karę $\lambda \sum w^2$. Gradient kary $2\lambda w$ ściąga wagi w stronę zera - preferowane są "gładsze" funkcje. W Keras: `kernel_regularizer=keras.regularizers.l2(lambda)`.
- **BatchNormalization:** normalizuje aktywacje w mini-batchu do średniej 0 i wariancji 1, potem skaluje uczonymi parametrami $\gamma, \beta$:
  $\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}},\; y = \gamma \hat{x} + \beta$.
  Głównie **stabilizuje i przyspiesza** trening (pozwala na większy lr); efekt regularyzujący jest uboczny (szum statystyk batcha) i słabszy niż Dropout.
- **Early stopping** - poznaliśmy w sekcji 3.

In [ ]:
# Mały zbiór treningowy - ułatwia przeuczenie
N_MALY = 500 if SZYBKI_TRYB else 1000
x_maly, y_maly = x_train[:N_MALY], y_train[:N_MALY]
EPOKI_REG = 3 if SZYBKI_TRYB else 60

def zbuduj_przeuczony():
    # Duży MLP bez regularyzacji: 784 -> 512 -> 512 -> 512 -> 10
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential([keras.Input(shape=(28, 28)), layers.Flatten()])
    for _ in range(3):
        model.add(layers.Dense(512, activation="relu"))
    model.add(layers.Dense(10, activation="softmax"))
    return model

def trenuj_reg(model, epoki=EPOKI_REG):
    # Wspólna procedura treningu dla wszystkich wariantów regularyzacji
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model.fit(x_maly, y_maly, epochs=epoki, batch_size=64,
                     validation_data=(x_val, y_val), verbose=0).history

model_over = zbuduj_przeuczony()
print("parametry:", model_over.count_params())
h_over = trenuj_reg(model_over)
rysuj_historie(h_over, tytul="Bez regularyzacji (mały zbiór)")
print(f"train acc: {h_over['accuracy'][-1]:.3f}, val acc: {h_over['val_accuracy'][-1]:.3f}, "
      f"min val_loss w epoce {int(np.argmin(h_over['val_loss'])) + 1}")

### Zadanie 5.1 - Dropout i L2

Napisz funkcję `zbuduj_regularyzowany(dropout=0.0, l2=0.0, batchnorm=False)`, która buduje taką samą architekturę jak `zbuduj_przeuczony`, ale:
- jeśli `l2 > 0`, każda warstwa ukryta `Dense` ma `kernel_regularizer=keras.regularizers.l2(l2)`,
- jeśli `batchnorm=True`, po każdej warstwie `Dense(512)` (bez aktywacji) dodaj `BatchNormalization()`, a następnie `Activation("relu")`,
- jeśli `dropout > 0`, po każdej warstwie ukrytej (po aktywacji) dodaj `Dropout(dropout)`.

Następnie wytrenuj funkcją `trenuj_reg` dwa warianty i zapisz historie:
- `h_drop` - Dropout 0.5,
- `h_l2` - L2 z $\lambda = 10^{-3}$.

**Kryteria sukcesu (pełny tryb):** dla obu wariantów rosnąca część `val_loss` jest wyraźnie mniejsza niż bez regularyzacji, a różnica między train i val accuracy maleje.

**Wskazówki:** pamiętaj o `keras.utils.set_random_seed(SEED)` na początku funkcji (uczciwe porównanie). Kolejność `Dense -> BatchNorm -> ReLU -> Dropout` to popularny wybór; w literaturze spotyka się też BN po aktywacji.

In [ ]:
# 5.1. Ten sam duzy MLP, tylko dokladamy regularyzacje.
def zbuduj_regularyzowany(dropout=0.0, l2=0.0, batchnorm=False):
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential([keras.Input(shape=(28, 28)), layers.Flatten()])
    reg = keras.regularizers.l2(l2) if l2 > 0 else None
    for _ in range(3):
        if batchnorm:
            model.add(layers.Dense(512, activation=None, kernel_regularizer=reg))
            model.add(layers.BatchNormalization())
            model.add(layers.Activation("relu"))
        else:
            model.add(layers.Dense(512, activation="relu", kernel_regularizer=reg))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(10, activation="softmax"))
    return model

h_drop = trenuj_reg(zbuduj_regularyzowany(dropout=0.5))
h_l2 = trenuj_reg(zbuduj_regularyzowany(l2=1e-3))
RAPORT["zad_5_1"] = {"dropout_val_acc_last": float(h_drop["val_accuracy"][-1]),
                      "l2_val_acc_last": float(h_l2["val_accuracy"][-1])}


### Zadanie 5.2 - BatchNormalization i zestawienie

1. Wytrenuj wariant z BatchNorm (`h_bn`) oraz wariant łączony BatchNorm + Dropout 0.3 (`h_bn_drop`).
2. Na jednym wykresie narysuj `val_loss` wszystkich pięciu wariantów (bez regularyzacji, Dropout, L2, BN, BN+Dropout).
3. Zbuduj tabelę: najlepsza `val_accuracy`, `val_accuracy` w ostatniej epoce, różnica `accuracy - val_accuracy` w ostatniej epoce (miara przeuczenia).
4. Sformułuj w markdown **hipotezę** przed uruchomieniem (np. "BN poprawi szybkość, ale słabiej ograniczy przeuczenie niż Dropout") i zweryfikuj ją po wynikach (UGŁ_U01).

**Kryteria sukcesu:** tabela i wykres kompletne, hipoteza zweryfikowana wynikami.

*Hipoteza przed treningiem:* BatchNorm powinien ustabilizować uczenie, a Dropout zmniejszyć lukę train–val. Sama BN nie musi ograniczyć przeuczenia równie mocno.


In [ ]:
# 5.2. Hipoteza przed trenowaniem jest nizej w notebooku. Tutaj juz same pomiary.
h_bn = trenuj_reg(zbuduj_regularyzowany(batchnorm=True))
h_bn_drop = trenuj_reg(zbuduj_regularyzowany(batchnorm=True, dropout=0.3))
warianty_reg = {
    "brak": h_over, "dropout_0.5": h_drop, "L2_1e-3": h_l2,
    "batchnorm": h_bn, "BN+dropout_0.3": h_bn_drop,
}
fig, ax = plt.subplots(figsize=(10, 5))
for nazwa, h in warianty_reg.items():
    ax.plot(range(1, len(h["val_loss"])+1), h["val_loss"], label=nazwa)
ax.set(xlabel="epoka", ylabel="val_loss", title="5.2 Regularyzacja - porównanie 5 wariantów")
ax.grid(alpha=0.25); ax.legend()
plt.tight_layout(); zapisz_wykres()
tabela_regularyzacji = pd.DataFrame([
    {"wariant": nazwa,
     "best_val_accuracy": float(max(h["val_accuracy"])),
     "last_val_accuracy": float(h["val_accuracy"][-1]),
     "last_train_minus_val": float(h["accuracy"][-1] - h["val_accuracy"][-1]),
     "val_loss_min": float(min(h["val_loss"])),
     "val_loss_last": float(h["val_loss"][-1])}
    for nazwa, h in warianty_reg.items()
])
print(tabela_regularyzacji.round(4).to_string(index=False))
tabela_regularyzacji.to_csv(WYNIKI / "regularyzacja.csv", index=False)
RAPORT["zad_5_2"] = tabela_regularyzacji.to_dict(orient="records")
# Porownuje konkrety, a nie to co chcialem zobaczyc przed testem.
print("Hipoteza BN vs Dropout: weryfikacja w tabeli powyzej (last_train_minus_val).")


*Weryfikacja hipotezy:* Porównaj wartości `last_train_minus_val` i `val_loss_last` w tabeli wygenerowanej powyżej. Wnioski z lokalnego wykonania są też zapisane w `wyniki/wnioski.txt` (nie przypisujemy oczekiwanych wyników przed uruchomieniem).


## 6. Zapis i wczytanie modelu

Wytrenowany model to element cyklu życia systemu (W1, W2): trzeba go zapisać, wersjonować, wczytać w innym procesie (np. serwis predykcyjny). W Keras 3:

| Co | Jak | Zawiera |
|---|---|---|
| pełny model | `model.save("plik.keras")`, `keras.models.load_model("plik.keras")` | architekturę, wagi, konfigurację `compile` i stan optymalizatora |
| tylko wagi | `model.save_weights("plik.weights.h5")`, `model.load_weights(...)` | tylko wartości wag - architekturę trzeba odtworzyć w kodzie |

Format `.keras` to archiwum zip (konfiguracja JSON + wagi). Starszy format `.h5` dla pełnych modeli oraz `SavedModel` są w Keras 3 przestarzałe. **Pamiętaj:** w Colab pliki zapisane na dysku maszyny wirtualnej znikają po zakończeniu sesji - aby je zachować, pobierz je lub zapisz na Google Drive.

Zapis wraz ze stanem optymalizatora pozwala **wznowić trening** dokładnie tam, gdzie go przerwano.

In [ ]:
# Demonstracja: zapis i wczytanie pełnego modelu
model_demo.save(str(MODELE / "model_demo.keras"))
wczytany = keras.models.load_model(str(MODELE / "model_demo.keras"))

p1 = model_demo.predict(x_test[:100], verbose=0)
p2 = wczytany.predict(x_test[:100], verbose=0)
print("Predykcje identyczne:", np.allclose(p1, p2, atol=1e-6))
print("Rozmiar pliku: %.1f kB" % (os.path.getsize(str(MODELE / "model_demo.keras")) / 1024))
print("Optymalizator wczytanego modelu:", type(wczytany.optimizer).__name__)

### Zadanie 6.1 - Zapis wag i wznowienie treningu

1. Zbuduj funkcją `zbuduj_regularyzowany(dropout=0.5)` model, wytrenuj go 2 epoki na `x_maly` (funkcja `trenuj_reg(model, epoki=2)`) i zapisz **same wagi** do `wagi_dropout.weights.h5`.
2. Zbuduj **nowy** model tą samą funkcją, wczytaj do niego wagi i sprawdź (`np.allclose`), że predykcje na `x_test[:100]` są identyczne. Pamiętaj, że Dropout w `predict` jest wyłączony - dlatego wynik jest deterministyczny.
3. Zapisz pierwszy model w całości do `model_dropout.keras`, wczytaj go i **kontynuuj trening** jeszcze 2 epoki przez `fit` (bez ponownego `compile`!). Wypisz `val_accuracy` przed i po.

**Kryteria sukcesu:** oba sprawdzenia `np.allclose` dają `True`; wczytany model trenuje się dalej bez błędu.

**Wskazówka:** nowy model z kroku 2 nie musi być skompilowany, żeby wczytać wagi i wykonać `predict`.

In [ ]:
# 6.1. Wagi i caly model to dwie rozne rzeczy. Sprawdzam obie.
model_do_zapisu = zbuduj_regularyzowany(dropout=0.5)
h_przed = trenuj_reg(model_do_zapisu, epoki=2)
sciezka_wag = MODELE / "wagi_dropout.weights.h5"
sciezka_model = MODELE / "model_dropout.keras"
model_do_zapisu.save_weights(sciezka_wag)
model_po_wagach = zbuduj_regularyzowany(dropout=0.5)
model_po_wagach.load_weights(sciezka_wag)
p_wagi_1 = model_do_zapisu.predict(x_test[:100], verbose=0)
p_wagi_2 = model_po_wagach.predict(x_test[:100], verbose=0)
wagi_zgodne = bool(np.allclose(p_wagi_1, p_wagi_2, atol=1e-6))
sprawdz(wagi_zgodne, "6.1: predykcje po wczytaniu wag takie same")
model_do_zapisu.save(sciezka_model)
model_wznowiony = keras.models.load_model(sciezka_model)
p_wszystko = model_wznowiony.predict(x_test[:100], verbose=0)
model_zgodny = bool(np.allclose(p_wagi_1, p_wszystko, atol=1e-6))
sprawdz(model_zgodny, "6.1: predykcje po wczytaniu pelnego modelu takie same")
h_po = model_wznowiony.fit(x_maly, y_maly, epochs=2, batch_size=64,
                            validation_data=(x_val, y_val), verbose=0)
acc_przed = float(h_przed["val_accuracy"][-1])
acc_po = float(h_po.history["val_accuracy"][-1])
print("Przed wznowieniem:", round(acc_przed, 4), "Po:", round(acc_po, 4))
RAPORT["zad_6_1"] = {"weights_predictions_equal": wagi_zgodne,
                       "full_model_predictions_equal": model_zgodny,
                       "val_acc_before_resume": acc_przed,
                       "val_acc_after_resume": acc_po,
                       "models_saved": [str(sciezka_wag), str(sciezka_model)]}


## Pytania kontrolne

1. Dlaczego w ostatniej warstwie klasyfikatora na 10 klas używamy softmax, a w regresji warstwy liniowej? Co by się stało, gdyby w regresji cen domów użyć sigmoidy?
2. Czym różni się `sparse_categorical_crossentropy` od `categorical_crossentropy`? Kiedy używamy której?
3. Dlaczego `StandardScaler` dopasowujemy tylko do zbioru treningowego? Jak nazywa się błąd polegający na dopasowaniu go do całych danych i jakie ma skutki?
4. Ile parametrów ma warstwa `Dense(64)` po wejściu o wymiarze 8? A `BatchNormalization` po niej (ile z nich jest uczonych gradientowo)?
5. Co robi `restore_best_weights=True` w `EarlyStopping` i dlaczego jest ważne?
6. Wyjaśnij, czym Adam różni się od zwykłego SGD. Dlaczego SGD bez momentum uczy się wolniej?
7. Na wykresie strata walidacyjna rośnie od 8. epoki, a treningowa dalej spada. Jak to nazywamy i jakie trzy działania podejmiesz?
8. Jak działa Dropout w trakcie treningu, a jak w predykcji? Dlaczego przeskalowuje się aktywacje przez $1/(1-p)$?

## Podsumowanie - co warto zapamiętać

- Keras: `Sequential` / `Input` -> `compile(optimizer, loss, metrics)` -> `fit(..., validation_data=...)` -> `evaluate` / `predict`.
- **Klasyfikacja:** wyjście `Dense(K, softmax)` + `sparse_categorical_crossentropy` (etykiety całkowite). **Regresja:** wyjście `Dense(1)` liniowe + MSE (raportuj MAE/RMSE).
- **Normalizuj dane;** parametry skalowania dopasowuj **tylko na train** - inaczej wyciek danych.
- Zawsze porównuj z **baseline'em** (średnia, regresja liniowa) - bez tego nie wiesz, czy sieć cokolwiek daje.
- **Krzywe uczenia** to podstawowe narzędzie diagnostyczne: niedouczenie vs przeuczenie.
- **Callbacki:** `EarlyStopping(restore_best_weights=True)`, `ModelCheckpoint(save_best_only=True)`, `ReduceLROnPlateau`.
- **Learning rate** jest najważniejszym hiperparametrem; Adam (1e-3) to dobry punkt startowy, SGD+momentum - solidna alternatywa.
- **Regularyzacja:** Dropout i L2 zmniejszają lukę train-val; BatchNorm głównie stabilizuje trening; najlepsze lekarstwo to więcej danych.
- Zapis: `.keras` (pełny model, wznowienie treningu) lub `.weights.h5` (same wagi).

## Do sprawozdania końcowego

Z tego laboratorium umieść w sprawozdaniu (zwięźle, maks. 1,5 strony):

1. **Krzywe uczenia** MLP z zad. 1.1 z krótką diagnozą (w której epoce zaczyna się przeuczenie) oraz dokładność testową.
2. **Tabelę regresji** z zad. 2.2 (średnia / regresja liniowa / MLP; MAE i RMSE) i wykres y vs y_pred z komentarzem o obciętych wartościach.
3. **Porównanie optymalizatorów** (zad. 4.1) - wykres i 2-3 zdania interpretacji.
4. **Tabelę regularyzacji** z zad. 5.2 wraz z **hipotezą i jej weryfikacją** (UGŁ_U01).
5. Jedną **propozycję usprawnienia** (UGŁ_U02), np. inny harmonogram lr, AdamW, augmentacja, inna architektura - z uzasadnieniem.

## Zadanie dodatkowe (dla chętnych) - własny callback i harmonogram lr

1. Napisz własny callback `ZatrzymajPrzyDokladnosci(keras.callbacks.Callback)`, który:
   - w `on_epoch_end` zapisuje do listy bieżący learning rate optymalizatora (`float(self.model.optimizer.learning_rate)`),
   - przerywa trening (`self.model.stop_training = True`), gdy `val_accuracy` przekroczy zadany próg (argument konstruktora), i wypisuje komunikat.
2. Wytrenuj MLP 256-128 na Fashion-MNIST z optymalizatorem Adam, którego learning rate jest harmonogramem `keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=...)` (liczba kroków = epoki x liczba batchy na epokę), z progiem 0.89.
3. Narysuj zapisany przez callback przebieg learning rate.

**Wskazówka:** liczba batchy na epokę to `ceil(len(x_train) / batch_size)`.

In [ ]:
EPOKI_DOD = 2 if SZYBKI_TRYB else 20
# BONUS. Callback sprawdza walidacje po epoce, a nie na treningu.
class ZatrzymajPrzyDokladnosci(keras.callbacks.Callback):
    def __init__(self, prog):
        super().__init__()
        self.prog = float(prog)
        self.learning_rates = []
        self.zatrzymano = False

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        obecny_lr = float(keras.ops.convert_to_numpy(self.model.optimizer.learning_rate))
        self.learning_rates.append(obecny_lr)
        accuracy = logs.get("val_accuracy")
        if accuracy is not None and accuracy >= self.prog:
            print(f"Próg {self.prog:.2f} osiągnięty: val_acc={accuracy:.3f}, epoka {epoch+1}")
            self.zatrzymano = True
            self.model.stop_training = True

keras.utils.set_random_seed(SEED)
model_dod = keras.Sequential([
    keras.Input(shape=(28, 28)), layers.Flatten(),
    layers.Dense(256, activation="relu"), layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
batch_dod = 128
kroki_epoka = int(np.ceil(len(x_train)/batch_dod))
schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3, decay_steps=EPOKI_DOD * kroki_epoka,
)
model_dod.compile(optimizer=keras.optimizers.Adam(learning_rate=schedule),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cb_prog = ZatrzymajPrzyDokladnosci(prog=0.89)
hist_dod = model_dod.fit(x_train, y_train, epochs=EPOKI_DOD, batch_size=batch_dod,
                         validation_data=(x_val, y_val), callbacks=[cb_prog], verbose=2)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(cb_prog.learning_rates)+1), cb_prog.learning_rates, "o-")
ax.set(xlabel="epoka", ylabel="learning rate", title="Bonus: CosineDecay i własny callback")
ax.grid(alpha=0.25)
plt.tight_layout(); zapisz_wykres()
RAPORT["bonus"] = {"zatrzymano_po_progu": cb_prog.zatrzymano,
                   "epochs": len(hist_dod.history["loss"]),
                   "best_val_accuracy": float(max(hist_dod.history["val_accuracy"]))}


In [ ]:
# Zapisuje finalne wyniki DOPIERO po wykonaniu wszystkich zadan.
import json
RAPORT["biblioteki"] = {"python": sys.version.split()[0], "numpy": np.__version__,
                         "keras": keras.__version__, "tensorflow": tf.__version__ if tf is not None else "brak",
                         "backend": keras.backend.backend()}
RAPORT["wykresy"] = [str(p).replace("\\", "/") for p in sorted(WYKRESY.glob("*.png"))]
RAPORT["ostrzezenia"] = OSTRZEZENIA
RAPORT["status"] = "OK - brak zgloszonych problemow" if not OSTRZEZENIA else "UWAGA - sprawdz ostrzezenia"
(WYNIKI / "podsumowanie.json").write_text(json.dumps(RAPORT, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

# Kilka wnioskow z FAKTYCZNYCH liczb; nie z samych oczekiwan.
mini = RAPORT["zad_1_1"]
reg = RAPORT["zad_2_2"]
proby = RAPORT["zad_5_2"]
bez = next(z for z in proby if z["wariant"] == "brak")
drop = next(z for z in proby if z["wariant"] == "dropout_0.5")
lr = RAPORT["zad_4_2"]
wnioski = [
    "Lab03 - wyniki lokalne (tryb: " + RAPORT["tryb"] + ")",
    f"1.1 Fashion-MNIST: acc na teście = {mini['test_accuracy']:.3f}.",
    f"1.2 Najwięcej pomyłek: {RAPORT['zad_1_2']['most_confused_pair'][0]} -> {RAPORT['zad_1_2']['most_confused_pair'][1]}.",
    f"2. California Housing: MAE średniej = {mae_sr:.3f}, linear = {mae_lin:.3f}, MLP = {mae_nn:.3f}.",
    f"3. Callbacki: epok {liczba_epok_cb}, najlepsza {najlepsza_epoka_cb}, MAE checkpointu = {mae_cb:.3f}.",
    f"4. Optim: najlepsze val_accuracy = {tabela_opt.iloc[0]['optymalizator']} ({tabela_opt.iloc[0]['best_val_accuracy']:.3f}).",
    "4.2 Adam LR: loss 0.0001 = " + str(lr["0.0001"]["loss_last"]) +
    ", 0.1 = " + str(lr["0.1"]["loss_last"]) + " (None=NaN/inf).",
    f"5. Overfitting: luka train-val bez regular. = {bez['last_train_minus_val']:.3f}, z dropout = {drop['last_train_minus_val']:.3f}.",
    f"6. Zapis: predykcje po wadze identyczne = {wagi_zgodne}; po pelnym modelu = {model_zgodny}.",
    "Usprawnienie na później: augmentacja obrazów Fashion-MNIST lub AdamW z porównaniem walidacji.",
    "UWAGA: dane Fashion-MNIST test tylko do raportu, dobór parametrów na walidacji.",
    "Ostrzezenia: " + ("; ".join(OSTRZEZENIA) if OSTRZEZENIA else "brak"),
]
(WYNIKI / "wnioski.txt").write_text("\n".join(wnioski) + "\n", encoding="utf-8")
print("\n=== LAB03 ZAKONCZONE ===")
print("Wyniki:", WYNIKI.resolve())
print("Liczba zapisanych wykresow:", len(RAPORT["wykresy"]))
print("Status:", RAPORT["status"])